# BareMetal-Transformer: train the `small` model on a free GPU

Trains the 6-layer / 512-d / 8-head byte-level GPT (~19M parameters) on TinyShakespeare with
`python_scratch/train_torch.py`, then downloads the checkpoint so you can export it for the C++ engine
(`python_scratch/export_weights.py`) on your own machine.

**Runtime → Change runtime type → GPU** (a free T4 is enough, ~15-20 min for 5000 steps).
On Kaggle: enable *Accelerator → GPU* and *Internet → On*, then run the same cells.

In [ ]:
import torch
print(torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

In [ ]:
# fresh clone each run; the repo only needs numpy + torch, both preinstalled on Colab/Kaggle
!rm -rf BareMetal_Transformers
!git clone https://github.com/SahilSahu73/BareMetal_Transformers.git
%cd BareMetal_Transformers
!pip -q install numpy

In [ ]:
# Sanity check before spending GPU time: the NumPy and PyTorch models must agree.
!pip -q install pytest
!python -m pytest python_scratch/test_model_parity.py -q

In [ ]:
# T4 has no native bfloat16 -> train in fp32. On an A100/L4 add --bf16 for ~2x speed.
!python python_scratch/train_torch.py --config small --steps 5000 --batch-size 32 \
    --lr 6e-4 --warmup 200 --eval-every 500 --out checkpoints/torch_small.pt

In [ ]:
# Download the best checkpoint (lowest validation loss) to your machine.
import torch
ck = torch.load("checkpoints/torch_small.pt", map_location="cpu")
print("step", ck["step"], "| val loss", round(ck["val_loss"], 4), "| config", ck["config"])
try:
    from google.colab import files
    files.download("checkpoints/torch_small.pt")
except ImportError:
    print("Not on Colab - on Kaggle, find the file in the Output panel (right sidebar).")

## Next, on your machine
```bash
mkdir -p checkpoints && mv ~/Downloads/torch_small.pt checkpoints/
python python_scratch/export_weights.py --ckpt checkpoints/torch_small.pt --output-dir checkpoints --name small --quantize-int8
python python_scratch/eval_quant.py --ckpt checkpoints/torch_small.pt
```
The exporter writes `small_fp32.bin`, `small_int8.bin`, `small_reference.bin` and `val_tokens.bin`.
